In [1]:
# Draft: decode Sporza indexed payload and map to existing PlayerInfo -> PlayerTable flow
from __future__ import annotations

from typing import Any, Dict, List

import requests

# Keep these imports if you run this notebook from the project root.
from models import PlayerInfo, PlayerTable

SPORZA_NEW_URL = "https://wielermanager.sporza.be/vrjr-m-26/competitions/5B2gvM7VF6.data?_routes=routes%2F%24edition.competitions.%24minicompetition.%28%24match%29"


def decode_indexed_payload(raw: List[Any]) -> Any:
    """
    Decode Sporza's flattened graph payload.

    Example token: {"_12": 13} means:
    - key is at raw[12]
    - value is at raw[13]
    """
    memo: Dict[int, Any] = {}
    in_progress: set[int] = set()

    def resolve_ref(ref: Any) -> Any:
        if isinstance(ref, int):
            return resolve_index(ref)
        return resolve_node(ref)

    def resolve_index(idx: int) -> Any:
        if idx in memo:
            return memo[idx]
        if idx in in_progress:
            # Prevent recursion loops if the graph has cycles.
            return None
        if idx < 0 or idx >= len(raw):
            return None

        in_progress.add(idx)
        result = resolve_node(raw[idx])
        memo[idx] = result
        in_progress.remove(idx)
        return result

    def resolve_node(node: Any) -> Any:
        if isinstance(node, dict):
            # Encoded object: all keys look like "_<index>"
            if node and all(isinstance(k, str) and k.startswith("_") and k[1:].isdigit() for k in node):
                out: Dict[str, Any] = {}
                for key_ref_token, value_ref in node.items():
                    key_ref = int(key_ref_token[1:])
                    key = resolve_index(key_ref)
                    value = resolve_ref(value_ref)
                    out[str(key)] = value
                return out

            # Fallback for regular objects (rare in this payload format).
            return {k: resolve_ref(v) for k, v in node.items()}

        if isinstance(node, list):
            # List entries are often integer refs.
            return [resolve_ref(item) for item in node]

        # Primitive: string, bool, None, etc.
        return node

    # In this payload, index 0 is the root object.
    return resolve_index(0)


def extract_player_info(decoded_root: Dict[str, Any]) -> List[PlayerInfo]:
    # Root shape is typically:
    # {"routes/$edition.competitions.$minicompetition.($match)": {"data": {...}}}
    route_payload = next(iter(decoded_root.values()))
    data = route_payload["data"]

    members = data["miniCompetition"]["members"]

    players: List[PlayerInfo] = []
    for member in members:
        team_name = member.get("teamName") or member.get("userName") or "Unknown"
        rank = int(member.get("rank", 0))
        points = int(member.get("points", 0))
        players.append(PlayerInfo(rank=rank, team=team_name, points=points))

    players.sort(key=lambda p: p.rank)
    return players


resp = requests.get(SPORZA_NEW_URL, timeout=25)
resp.raise_for_status()
raw_payload = resp.json()

if not isinstance(raw_payload, list):
    raise ValueError(f"Expected list payload, got {type(raw_payload).__name__}")

decoded_root = decode_indexed_payload(raw_payload)
players = extract_player_info(decoded_root)

print(f"Mapped teams: {len(players)}")

if not players:
    raise ValueError("No members were mapped from decoded payload")

table = PlayerTable(players)
print(table.create_table_string())



Mapped teams: 10
```
# | Team                 | Points
---------------------------------
1 | Voldoende Chamois    | 0     
2 | D’r op en d’r over   | 0     
3 | MekaniekFysiek       | 0     
4 | DE Leanderleetjes    | 0     
5 | Pakje Voor Pamela    | 0     
6 | Will the real WVA    | 0     
7 | Riebe De Bie         | 0     
8 | Carry                | 0     
9 | Del van Café Sport   | 0     
10 | Rialcé               | 0     
Last update: 28-02-26 - 12:29:34
```
